# Phase 3: Model Training and Anomaly Score Distributions

In this notebook, we visualize the anomaly scores generated by our three models:
1. **Isolation Forest** (trained on statistical features)
2. **One-Class SVM** (trained on statistical features)
3. **1D Conv Autoencoder** (trained on raw vibration signals)

A good anomaly detector will assign *low* scores to normal data, and *high* scores to fault data.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

## Load Anomaly Scores
These scores were generated by `src/baselines.py` and `src/autoencoder.py`.

In [ ]:
data_dir = "../data/processed"
scores_dir = os.path.join(data_dir, "scores")

try:
    if_scores = np.load(os.path.join(scores_dir, "if_scores.npy"))
    ocsvm_scores = np.load(os.path.join(scores_dir, "ocsvm_scores.npy"))
    ae_scores = np.load(os.path.join(scores_dir, "ae_scores.npy"))
    meta = pd.read_csv(os.path.join(data_dir, "metadata.csv"))
    
    df = meta.copy()
    df['IF_Score'] = if_scores
    df['OCSVM_Score'] = ocsvm_scores
    df['AE_Score'] = ae_scores
    
    print("Scores loaded successfully!")
    display(df.head())
except FileNotFoundError:
    print("Error: Scores not found. Make sure to run src/baselines.py and src/autoencoder.py first.")

## Isolation Forest Anomaly Scores

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='fault_type', y='IF_Score', hue='fault_type')
plt.title("Isolation Forest Anomaly Score by Fault Type")
plt.ylabel("Anomaly Score (Higher is more anomalous)")
plt.show()

## Autoencoder Reconstruction Error
For the Autoencoder, the "anomaly score" is simply the Mean Squared Error (MSE) between the input signal and the reconstructed signal.

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='fault_type', y='AE_Score', hue='fault_type')
plt.title("Autoencoder Reconstruction Error by Fault Type")
plt.ylabel("Reconstruction Error (MSE)")
plt.yscale('log') # Log scale because errors can span multiple orders of magnitude
plt.show()